In [1]:
import sys
sys.path.append("..")            # 저장소 루트 (project 패키지)
sys.path.append("../scripts")    # eval_silver 변환 함수 재사용
from pathlib import Path

In [2]:
BENCH_ID = "AIHub_KorLectureSpeech_lecture_clean"
SILVER   = "/data/ASR/BENCHMARK/SILVER/AIHub_KorLectureSpeech/transcript.jsonl"
MODEL    = "openai/whisper-small"
DEVICE   = "cuda:2"              # 실행 직전 nvidia-smi 로 빈 GPU 확인
SAMPLE   = 2000                  # 무작위 샘플 크기 (전체 32,971 중)
SEED     = 42                    # 재현용

OUT_DIR  = Path(f"../BENCHMARK/results/whisper_small__{BENCH_ID}")

In [3]:
import random
from eval_silver import convert_silver

# 1) 전체 변환 (원본 → GOLD 필드명). 원본 SILVER 는 읽기만 함.
conv_full = OUT_DIR / "_silver_converted" / f"{BENCH_ID}.jsonl"
n = convert_silver(Path(SILVER), conv_full, corpus_id=BENCH_ID)
print(f"전체 변환: {n} samples")

# 2) 무작위 SAMPLE개 추출 (seed 고정 → 재현 가능)
lines = conv_full.read_text(encoding="utf-8").splitlines()
random.seed(SEED)
sample_lines = random.sample(lines, SAMPLE)
conv = conv_full.with_name(f"{BENCH_ID}__sample{SAMPLE}.jsonl")
conv.write_text("\n".join(sample_lines) + "\n", encoding="utf-8")
print(f"무작위 샘플: {len(sample_lines)} samples → {conv}")

전체 변환: 32971 samples
무작위 샘플: 2000 samples → ../BENCHMARK/results/whisper_small__AIHub_KorLectureSpeech_lecture_clean/_silver_converted/AIHub_KorLectureSpeech_lecture_clean__sample2000.jsonl


In [4]:
from project.data.adapters.whisper import build_predict_fn

predict_fn = build_predict_fn(
    MODEL, backbone=MODEL,
    language="ko", task="transcribe",
    beam_size=5, batch_size=16, device=DEVICE,
)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [5]:
from project.evaluation import evaluate_on_benchmark_suite

results = evaluate_on_benchmark_suite(
    model_name=f"whisper_small__{BENCH_ID}",
    predict_fn=predict_fn,
    benchmark_paths={BENCH_ID: conv},
    out_dir=OUT_DIR,
    batch_size=16,
)
results[BENCH_ID]

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take p

CerResult(cer=18.266482869918974, scer=18.711320363625596, wer=45.290665766032, samples=2000, per_sample_cer=[25.0, 9.523809523809524, 28.000000000000004, 0.0, 17.307692307692307, 17.391304347826086, 45.45454545454545, 33.33333333333333, 14.705882352941178, 16.304347826086957, 15.789473684210526, 25.862068965517242, 9.900990099009901, 12.307692307692308, 10.0, 50.0, 0.0, 12.76595744680851, 3.260869565217391, 39.130434782608695, 11.11111111111111, 38.666666666666664, 0.0, 36.84210526315789, 16.666666666666664, 7.352941176470589, 0.0, 26.190476190476193, 12.0, 8.333333333333332, 8.641975308641975, 16.0, 15.384615384615385, 9.782608695652174, 5.357142857142857, 5600.0, 40.0, 66.66666666666666, 6.730769230769231, 3.8461538461538463, 9.30232558139535, 20.0, 7.8125, 16.666666666666664, 13.432835820895523, 29.78723404255319, 7.017543859649122, 0.0, 9.090909090909092, 25.0, 10.975609756097562, 6.25, 21.34831460674157, 35.714285714285715, 11.11111111111111, 4.3478260869565215, 50.0, 57.14285714

In [6]:
print((OUT_DIR / "evaluation_report.txt").read_text(encoding="utf-8"))

📊 ASR Evaluation Report — whisper_small__AIHub_KorLectureSpeech_lecture_clean
   Date: 2026-06-16T17:55:56

## 1. Benchmark Set Results (한국어 CER 표준)
--------------------------------------------------------------------------------
Benchmark                                                  CER (%)   sCER (%)    Samples
--------------------------------------------------------------------------------
AIHub_KorLectureSpeech_lecture_clean                         18.27      18.71      2,000
--------------------------------------------------------------------------------
Weighted Average                                             18.27                 2,000

## 2. Slice Analysis (메타 필드별)
--------------------------------------------------------------------------------

### AIHub_KorLectureSpeech_lecture_clean
  [by age_group]
  value                   CER (%)    samples
  70대                       29.65         35
  unknown                   21.87        651
  50대                       21.48  

In [7]:
import json
import pandas as pd
import jiwer

lines = (OUT_DIR / BENCH_ID / "predictions.jsonl").read_text(encoding="utf-8").splitlines()
df = pd.DataFrame(json.loads(l) for l in lines if l)

df["cer"] = [
    jiwer.cer(r, h) * 100 if r else float("nan")
    for r, h in zip(df["text_normalized"], df["prediction_normalized"])
]

for _, row in df.sort_values("cer", ascending=False).head(20).iterrows():
    print(f"[CER {row.cer:5.1f}] 정답: {row.text_normalized}")
    print(f"             예측: {row.prediction_normalized}\n")

[CER 5600.0] 정답: 에이는 에이는
             예측: 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는 에이는

[CER 1100.0] 정답: 근데 세계 이 시장의 에 에서만,
             예측: 그런데 세계 시장에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에에

[CER 300.0] 정답: 백
             예측: 100

[CER 240.0] 정답: 대통령이.
             예측: mbc 뉴스 이준범입니다

[CER 200.0] 정답: 그 엄청나죠
             예측: 엄청난 일 아닙니까 그거 엄청나죠

[CER 152.9] 정답: 그래서 에이가 두 번 뽑히고 비가 세 번 뽑히고 씨가 두 번 뽑혔다는 뜻은 이 값은 이컴마삼컴마 이라는 글을 갖는다는 말로 이해하면 되는 겁니다. 따라서 서로 다른 숫자에이 비 (C)(씨
             예측: a가 두 